# UFC Fight Winner Prediction Pipeline

This notebook provides a modular framework to train, evaluate, and use Machine Learning and Deep Learning models to predict UFC fight outcomes. 

### Features:
1. **Modular Design**: A `UFCFightPredictor` class handles data loading, preprocessing, training, and prediction.
2. **Leakage Prevention**: Automatically removes columns that contain post-fight information.
3. **Model Comparison**: Compares XGBoost, Random Forest, Gradient Boosting, Logistic Regression, and a PyTorch Neural Network.
4. **Unseen Data**: Includes a method to ingest a new CSV file and generate predictions using trained models.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
import copy

# Set device for PyTorch
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [2]:
class DeepUFCNet(nn.Module):
    """
    A flexible Neural Network architecture for tabular data.
    """
    def __init__(self, input_dim, hidden_layers=[128, 64]):
        super(DeepUFCNet, self).__init__()
        layers = []
        in_dim = input_dim
        
        for h_dim in hidden_layers:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.3))
            in_dim = h_dim
        
        layers.append(nn.Linear(in_dim, 1))
        layers.append(nn.Sigmoid())
        
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [3]:
class UFCFightDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32).to(device)
        self.y = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1).to(device) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

In [4]:
class UFCFightPredictor:
    def __init__(self):
        self.models = {}
        self.scaler = StandardScaler()
        self.label_encoders = {}
        self.numeric_cols = []
        self.categorical_cols = ['BlueStance', 'RedStance', 'BetterRank']
        self.feature_columns = [] # To ensure column order matches during prediction
        
        # Define Sklearn/XGB models
        self.ml_models_config = {
            'XGBoost': XGBClassifier(n_estimators=1000, learning_rate=0.01, max_depth=5, 
                                     subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42),
            'LogisticRegression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
            'RandomForest': RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
            'GradientBoosting': GradientBoostingClassifier(n_estimators=200, random_state=42)
        }

    def _preprocess_data(self, df, fit=False):
        """
        Internal method to clean and prepare data. 
        If fit=True, it learns scalers/encoders (training mode).
        If fit=False, it applies existing scalers/encoders (prediction mode).
        """
        df = df.copy()
        
        # 1. Target Engineering (only if training)
        if 'Winner' in df.columns and fit:
            df['BlueWin'] = (df['Winner'] == 'Blue').astype(int)
            y = df['BlueWin']
        else:
            y = None

        # 2. Feature Selection
        # Define exclusions (ID columns and Data Leakage columns)
        exclude_cols = ['Winner', 'BlueWin', 'RedFighter', 'BlueFighter', 'Date', 
                        'Location', 'Country', 'TitleBout', 'EmptyArena', 'WeightClass']
        leak_keywords = ['Finish', 'TotalFightTime', 'Referee', 'Round', 'weightRank', 'FPRank', 'WinsBy']
        
        # Identify numeric columns dynamically if fitting
        if fit:
            all_cols = df.columns.tolist()
            self.feature_columns = [c for c in all_cols if c not in exclude_cols 
                                    and not any(k in c for k in leak_keywords)]
            
            self.numeric_cols = []
            for c in self.feature_columns:
                if c in self.categorical_cols:
                    continue
                if pd.api.types.is_numeric_dtype(df[c]):
                    self.numeric_cols.append(c)
        
        # Ensure we only work with selected features
        X = df[self.feature_columns].copy()

        # 3. Handle Categorical
        for col in self.categorical_cols:
            if col in X.columns:
                X[col] = X[col].fillna('Unknown').astype(str)
                if fit:
                    le = LabelEncoder()
                    X[col] = le.fit_transform(X[col])
                    self.label_encoders[col] = le
                else:
                    # Handle unseen labels by mapping to a default or encoding as-is if possible
                    # Simple approach: Use known classes, fill unknown with mode or 0
                    le = self.label_encoders[col]
                    X[col] = X[col].map(lambda s: s if s in le.classes_ else le.classes_[0])
                    X[col] = le.transform(X[col])

        # 4. Handle Numeric (Fill NaN with 0 for debutant logic)
        X[self.numeric_cols] = X[self.numeric_cols].fillna(0)

        # 5. Log Transform Skewed Features
        skewed_cols = ['BlueTotalRoundsFought', 'RedTotalRoundsFought', 'BlueTotalTitleBouts', 'RedTotalTitleBouts']
        for col in skewed_cols:
            if col in X.columns:
                X[col] = np.log1p(X[col])

        # 6. Scale Numeric Features
        if fit:
            X[self.numeric_cols] = self.scaler.fit_transform(X[self.numeric_cols])
        else:
            X[self.numeric_cols] = self.scaler.transform(X[self.numeric_cols])

        return X, y

    def train(self, csv_path):
        print(f"Loading data from {csv_path}...")
        df = pd.read_csv(csv_path)
        
        print("Preprocessing data...")
        X, y = self._preprocess_data(df, fit=True)
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        
        self.results = []

        # --- Train ML Models ---
        for name, model in self.ml_models_config.items():
            print(f"Training {name}...")
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            acc = accuracy_score(y_test, preds)
            f1 = f1_score(y_test, preds)
            self.models[name] = model
            self.results.append({'Model': name, 'Accuracy': acc, 'F1-Score': f1})

        # --- Train Neural Network ---
        print("Training Neural Network (PyTorch)...")
        train_dataset = UFCFightDataset(X_train.values, y_train)
        test_dataset = UFCFightDataset(X_test.values, y_test)
        
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
        
        nn_model = DeepUFCNet(input_dim=X_train.shape[1]).to(device)
        criterion = nn.BCELoss()
        optimizer = optim.AdamW(nn_model.parameters(), lr=0.001)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

        best_acc = 0
        epochs = 20
        
        for epoch in range(epochs):
            nn_model.train()
            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()
                outputs = nn_model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
            
            # Validation
            nn_model.eval()
            with torch.no_grad():
                val_X = test_dataset.X
                val_y = test_dataset.y
                val_outputs = nn_model(val_X)
                val_preds = (val_outputs > 0.5).float()
                val_acc = accuracy_score(val_y.cpu(), val_preds.cpu())
                
            scheduler.step(val_acc)
            if val_acc > best_acc:
                best_acc = val_acc
                best_model_state = copy.deepcopy(nn_model.state_dict())
        
        # Load best NN state
        nn_model.load_state_dict(best_model_state)
        self.models['NeuralNetwork'] = nn_model
        self.results.append({'Model': 'NeuralNetwork', 'Accuracy': best_acc, 'F1-Score': 'N/A'})

        print("Training Complete.")

    def evaluate_models(self):
        """Returns a DataFrame comparing model performance."""
        return pd.DataFrame(self.results).sort_values(by='Accuracy', ascending=False)

    def predict_new_data(self, new_csv_path):
        """
        Predicts winners for a new dataset using the best trained model (highest accuracy).
        """
        if not self.models:
            raise ValueError("Models not trained. Run .train() first.")
            
        print(f"Loading new data from {new_csv_path}...")
        new_df = pd.read_csv(new_csv_path)
        
        # Preprocess using stored scaler/encoders
        X_new, _ = self._preprocess_data(new_df, fit=False)
        
        # Select best model based on training accuracy
        best_model_name = max(self.results, key=lambda x: x['Accuracy'])['Model']
        print(f"Using best model: {best_model_name}")
        
        model = self.models[best_model_name]
        
        if best_model_name == 'NeuralNetwork':
            model.eval()
            with torch.no_grad():
                X_tensor = torch.tensor(X_new.values, dtype=torch.float32).to(device)
                preds_proba = model(X_tensor).cpu().numpy()
                preds = (preds_proba > 0.5).astype(int).flatten()
        else:
            preds = model.predict(X_new)
            preds_proba = model.predict_proba(X_new)[:, 1]

        # Create Result DataFrame
        results = new_df[['RedFighter', 'BlueFighter']].copy()
        results['PredictedWinner'] = ['Blue' if p == 1 else 'Red' for p in preds]
        results['BlueWinProbability'] = preds_proba
        
        return results

## 1. Train and Evaluate Models

In [5]:
# Initialize and Run
predictor = UFCFightPredictor()

# Train on the master dataset
# Ensure 'ufc-master.csv' is in the same directory
predictor.train('ufc-master.csv')

Loading data from ufc-master.csv...
Preprocessing data...
Training XGBoost...


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:Gender: object

In [ ]:
# Compare Model Performance
performance_df = predictor.evaluate_models()
print("Model Performance Comparison:")
display(performance_df)

## 2. Predict on Unseen Data
To test on new data, create a CSV file (e.g., `upcoming_fights.csv`) with the same columns as `ufc-master.csv` (minus the results). The code below simulates this by taking a sample from the master file.

In [ ]:
# Simulate unseen data by saving a sample to a new CSV
# sample_data = pd.read_csv('ufc-master.csv').sample(5, random_state=99)
# # In a real scenario, you would drop the 'Winner' column as you wouldn't know it yet
# unseen_data = sample_data.drop(columns=['Winner'])
# unseen_data.to_csv('upcoming_fights.csv', index=False)

# print("Created dummy 'upcoming_fights.csv' for testing.")

# Run Prediction
predictions = predictor.predict_new_data('upcoming.csv')

print("\nPredictions for Upcoming Fights:")
display(predictions)